# 03 · Калибровка, усреднение фолдов и независимый test
## Кратко
По train-CV выбрана **D_hierarchy**, по validation — **single / raw**. Её test macro-F1 **0.8656**, recall pathologic **91.4%**, AP **0.9264**. Групповой 95% интервал macro-F1: **0.768–0.922**. В test только 35 патологических сегментов из 10 файлов.

Validation-разница ensemble − single по macro-F1: 0.0066; по правилу ≥0.01 выбран single. На test обычный MultiClass имеет более высокий macro-F1 (0.8792); чемпион по этому результату не переименовывается. Это наблюдение показывает неопределённость ранжирования, а не повод снова подбирать модель по test.

## Контекст и методы
Каждый финальный CatBoost обучается на train, не на validation. Для проверки postprocessing на validation используются общие 3 групповых cross-fitting-фолда: calibrator и threshold fit на двух частях, оценка на третьей. Калибратор ансамбля обучается на усреднённых выходах ансамбля, отдельно от single. Ensemble выбирается только при приросте macro-F1 не менее 0.01 на validation cross-fit; это заранее заданное практическое правило.

### Ключевые допущения
В validation лишь 29 патологических сегментов; разница небольших метрик шумна. После выбора калибратор и порог fit на всей validation, деревья не переобучаются на ней. Иначе уже оценённая калибровка относилась бы к другой модели. `locked_decision.json` сохраняется до первого test predict. Интервалы на test получаются 1000-повторным bootstrap исходных файлов и условны на уже обученной модели.

In [3]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.training.ctg_experiments import OUT, MODELS, load_data, FAMILIES, CTG_FEATURES
sns.set_theme(style='whitegrid', palette=['#245b87', '#c47c2d', '#80659e'])
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})
data, fingerprint = load_data()

## Данные и выбранное в 02 семейство

In [5]:
selection = json.loads((OUT / 'nested_selection.json').read_text())
print('Семейство по nested CV:', selection['family'])
display(data.groupby('split').agg(segments=('NSP','size'), recordings=('source_file','nunique')))
print('Test не участвует ни в одном fit; это аудитируемый локальный benchmark, не новая клиническая когорта.')

Семейство по nested CV: D_hierarchy
Test не участвует ни в одном fit; это аудитируемый локальный benchmark, не новая клиническая когорта.


,segments,recordings
split,,
test,425,68
train,1360,227
validation,341,57


## Финальный поиск, validation cross-fitting и заблокированный test

In [7]:
from src.training.ctg_experiments import run_final
test_results, validation_results, intervals = run_final()

final grid · A_multiclass: 23.6s
final grid · B_sqrt: 24.8s
final grid · B_balanced: 26.0s
final grid · C_ovr: 39.5s
final grid · D_hierarchy: 23.1s
final grid · logistic: 0.6s


### Что показала validation

In [9]:
display(validation_results[['family','architecture','mode','macro_f1','recall_path','ap_path','log_loss','brier']].round(4))
locked = json.loads((OUT / 'locked_decision.json').read_text())
print('Выбран:', locked['family'], '/', locked['architecture'])
print('Критерий ensemble: прирост macro-F1 ≥',locked['ensemble_minimum_gain'])

Выбран: D_hierarchy / single
Критерий ensemble: прирост macro-F1 ≥ 0.01


,family,architecture,mode,macro_f1,recall_path,ap_path,log_loss,brier
0,A_multiclass,single,raw,0.8161,0.8966,0.8705,0.2497,0.1386
1,A_multiclass,single,calibrated,0.8300,0.8276,0.8890,0.2427,0.1348
2,A_multiclass,single,threshold,0.8357,0.8966,0.8890,0.2427,0.1348
3,B_sqrt,single,raw,0.8449,0.8966,0.8596,0.2392,0.1321
4,B_sqrt,single,calibrated,0.8449,0.8621,0.8838,0.2226,0.1218
5,B_sqrt,single,threshold,0.8449,0.8621,0.8838,0.2226,0.1218
6,B_balanced,single,raw,0.8220,0.8966,0.8375,0.2889,0.1606
7,B_balanced,single,calibrated,0.8194,0.7931,0.8577,0.2295,0.1302
8,B_balanced,single,threshold,0.7996,0.7586,0.8577,0.2295,0.1302
9,C_ovr,single,raw,0.8283,0.9310,0.8467,0.3009,0.1509


### Финальная оценка test · чемпион выбран заранее

In [11]:
display(test_results[['family','architecture','mode','champion','accuracy','macro_f1','balanced_accuracy','recall_path','precision_path','ap_path','log_loss','brier','path_as_normal']].round(4))
champion = test_results[test_results.champion].iloc[0]
shown = intervals.loc[['macro_f1','recall_path','precision_path','ap_path','log_loss','brier']].copy()
shown.insert(0,'estimate',[champion[name] for name in shown.index])
display(shown.round(4))
print('Случаев pathologic в test:', (data.loc[data.split=='test','NSP']==3).sum())

Случаев pathologic в test: 35


,family,architecture,mode,champion,accuracy,macro_f1,balanced_accuracy,recall_path,precision_path,ap_path,log_loss,brier,path_as_normal
0,A_multiclass,single,threshold,False,0.9365,0.8792,0.8620,0.9143,0.8649,0.9271,0.2062,0.1098,0.0286
1,B_sqrt,single,calibrated,False,0.9271,0.8657,0.8579,0.9143,0.8421,0.9207,0.2039,0.1078,0.0286
2,B_balanced,single,raw,False,0.9082,0.8492,0.9068,0.9143,0.7442,0.9199,0.2512,0.1391,0.0286
3,C_ovr,single,calibrated,False,0.9224,0.8482,0.8209,0.8286,0.8286,0.9203,0.2269,0.1196,0.1143
4,D_hierarchy,single,raw,True,0.9271,0.8656,0.8627,0.9143,0.8205,0.9264,0.2253,0.1206,0.0286
5,D_hierarchy,fold_ensemble,calibrated,False,0.9247,0.8490,0.8171,0.8286,0.8529,0.9258,0.2061,0.1132,0.1143
6,logistic,single,calibrated,False,0.9012,0.7915,0.7758,0.7714,0.7297,0.8416,0.2654,0.1516,0.1429
7,always_normal,constant,raw,False,0.7812,0.2924,0.3333,0.0000,0.0000,0.0824,0.6704,0.3644,1.0000


,estimate,lower,upper
macro_f1,0.8656,0.7676,0.9218
recall_path,0.9143,0.6957,1.0000
precision_path,0.8205,0.5623,0.9412
ap_path,0.9264,0.7730,0.9814
log_loss,0.2253,0.1450,0.3158
brier,0.1206,0.0762,0.1734


### Экспорт и проверка сохранённого артефакта

In [13]:
import joblib
bundle = joblib.load(MODELS / 'champion.joblib')
example = data.loc[data.split=='test',list(CTG_FEATURES)].head(3)
probability = bundle.predict_proba(example)
assert np.allclose(probability.sum(axis=1),1)
display(pd.DataFrame(probability,columns=['P(normal)','P(suspect)','P(pathologic)']))
print('Прогнозы NSP:',bundle.predict(example).tolist())
print('Модель:',MODELS/'champion.joblib')
print('Происхождение, версии и правила:',MODELS/'model_card.json')

Прогнозы NSP: [1, 1, 3]
Модель: /Users/nick/Desktop/projects/femfedora/models/exported/ctg_uci/champion.joblib
Происхождение, версии и правила: /Users/nick/Desktop/projects/femfedora/models/exported/ctg_uci/model_card.json


,P(normal),P(suspect),P(pathologic)
0,0.690665,0.302309,0.007026
1,0.988123,0.011712,0.000165
2,0.358038,0.030457,0.611506


## Выводы
Таблица test служит финальной оценкой зафиксированных процедур; она не меняет выбор чемпиона. AP — Average Precision патологического класса; Brier — средняя сумма квадратов ошибки трёх вероятностей (диапазон 0–2). Accuracy следует читать рядом с recall/AP патологии и normal-baseline. Усреднение фолдов принимается только при подтверждённом на validation выигрыше; наличие пяти моделей само по себе не доказывает улучшения. Для эксплуатационного применения требуется проверка на другом источнике и данных с реальными исходами.